# Notebook 04: Inference

Adds to the original K2N-holdout / N2K-test evaluation:
- image_id tracking on every saved prediction (required for cluster bootstrap)
- in-domain baselines: K2N checkpoints -> Kermany test (K2K), N2K checkpoints -> seed-matched Togunwa holdout (N2N)
- supplementary K2N checkpoints -> full 190-image Togunwa set (K2N_full), kept alongside (not replacing) the 30-image holdout-only evaluation

The 30-image Togunwa holdout is carved out before kfold_splits runs (see notebook 02), so it is doubly out-of-sample for N2K models: never used for training, and never used for early-stopping/model-selection either. This makes it a valid shared evaluation target for both K2N and N2N.

## 1. Setup

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
import pandas as pd
import numpy as np
import torch

import config
from src import splits as SP
from src import datasets as DS
from src import models as M
from src import inference as INF
from src import metrics as MET
from src import results as RES

for mod in (config, SP, DS, M, INF, MET, RES):
    importlib.reload(mod)

config.ensure_output_dirs()
device = config.get_device()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Compute Device: GPU (Tesla T4)


In [3]:
PRED_DIR = config.RESULTS_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = config.RESULTS_DIR / "manifests"
manifest = pd.read_csv(MANIFEST_DIR / "splits_all.csv")
eval_targets = pd.read_csv(MANIFEST_DIR / "eval_targets.csv")

tf_eval = DS.build_transforms(config.IMAGE_SIZE, config.IMAGENET_MEAN, config.IMAGENET_STD, train=False)

print(f"Manifest shape: {manifest.shape} | Eval targets shape: {eval_targets.shape}")


Manifest shape: (7644, 10) | Eval targets shape: (624, 10)


## 2. Evaluation target builders

In [4]:
def build_togunwa_holdout_ds(seed):
    """30-image Togunwa holdout for a given seed. Never trained on by K2N
    (no Togunwa data at all) or N2K (excluded before kfold_splits runs)."""
    rows = manifest[
        (manifest.dataset == "togunwa") &
        (manifest.split == "holdout") &
        (manifest.seed.astype(str) == str(seed))
    ].reset_index(drop=True)
    ds = DS.CXRDataset(rows[["path", "label"]], tf_eval)
    return ds, rows["path"].tolist()


def build_togunwa_full_ds():
    """All 190 Togunwa images. Safe for K2N models only (they never see any
    Togunwa data during training, so there is no leakage risk)."""
    records = SP.collect_records(config.TOGUNWA_CLASSES, config.CLASS_TO_IDX)
    df = pd.DataFrame(records)[["path", "label"]]
    ds = DS.CXRDataset(df, tf_eval)
    return ds, df["path"].tolist()


def build_kermany_test_ds():
    """Fixed 624-image Kermany test set, used for both N2K (existing) and
    the new K2K in-domain baseline."""
    rows = eval_targets[
        (eval_targets.dataset == "kermany") & (eval_targets.split == "test")
    ].reset_index(drop=True)
    ds = DS.CXRDataset(rows[["path", "label"]], tf_eval)
    return ds, rows["path"].tolist()


kermany_test_dataset, kermany_test_paths = build_kermany_test_ds()
print(f"\nKermany test set size: {len(kermany_test_paths)} (used for N2K and K2K)")

togunwa_full_dataset, togunwa_full_paths = build_togunwa_full_ds()
print(f"Togunwa full set size: {len(togunwa_full_paths)} (used for K2N_full)")

for seed in config.SEEDS:
    _, holdout_paths = build_togunwa_holdout_ds(seed)
    print(f"Togunwa holdout seed {seed} size: {len(holdout_paths)} (used for K2N and N2N)")



Kermany test set size: 624 (used for N2K and K2K)
Togunwa full set size: 190 (used for K2N_full)
Togunwa holdout seed 42 size: 30 (used for K2N and N2N)
Togunwa holdout seed 43 size: 30 (used for K2N and N2N)
Togunwa holdout seed 44 size: 30 (used for K2N and N2N)
Togunwa holdout seed 45 size: 30 (used for K2N and N2N)
Togunwa holdout seed 46 size: 30 (used for K2N and N2N)
Togunwa holdout seed 47 size: 30 (used for K2N and N2N)


## 3. Inference runner (writes image_id-tagged prediction CSVs)

In [5]:
def run_inference_and_save(model, dataset, image_paths, tag, target_name):
    """Runs inference, attaches image_id, saves CSV, returns metrics dict.
    image_id = the on-disk path, which is stable and unique per image; dataset
    order matches image_paths exactly because CXRDataset does not shuffle."""
    out = INF.predict(model, dataset, device, batch_size=32, temperature=1.0, num_workers=2)

    pred_df = pd.DataFrame({
        "image_id": image_paths,
        "y_true": out["y_true"],
        "y_prob": out["y_prob"],
        "logit_0": out["logits"][:, 0],
        "logit_1": out["logits"][:, 1],
    })
    out_csv_path = PRED_DIR / f"{tag}__on__{target_name}.csv"
    pred_df.to_csv(out_csv_path, index=False)

    n_bins = MET.n_bins_for_sample_size(len(out["y_true"]), config.ECE_MIN_BINS, config.ECE_MAX_BINS)
    m = MET.all_metrics(out["y_true"], out["y_prob"], fixed_sensitivity=config.FIXED_SENSITIVITY, n_bins=n_bins)
    m["n_bins_used"] = n_bins
    return m, out_csv_path


In [6]:
checkpoints = sorted(config.CHECKPOINTS_DIR.glob("*.pth"))
print(f"\nFound {len(checkpoints)} checkpoint file(s) for evaluation.")

records = []

for ckpt in checkpoints:
    info = RES.parse_run_id(ckpt.stem)
    arch = info["arch"]
    direction = info["direction"]
    seed = info["seed"]

    model = M.build_model(arch, pretrained=False)
    model.load_state_dict(torch.load(ckpt, map_location=device))

    if direction == "K2N":
        # K2N (Togunwa holdout, seed-matched)
        holdout_ds, holdout_paths = build_togunwa_holdout_ds(seed)
        m, _ = run_inference_and_save(model, holdout_ds, holdout_paths, ckpt.stem, "togunwa_holdout")
        records.append({**info, "target": "togunwa_holdout",
                         "condition": RES.derive_condition("K2N", "togunwa_holdout"), **m})

        # K2K in-domain baseline (Kermany test)
        m2, _ = run_inference_and_save(model, kermany_test_dataset, kermany_test_paths, ckpt.stem, "kermany_test")
        records.append({**info, "target": "kermany_test",
                         "condition": RES.derive_condition("K2N", "kermany_test"), **m2})

        # Supplementary: K2N_full (all 190 Togunwa images, no leakage risk)
        m3, _ = run_inference_and_save(model, togunwa_full_dataset, togunwa_full_paths, ckpt.stem, "togunwa_full")
        records.append({**info, "target": "togunwa_full",
                         "condition": RES.derive_condition("K2N", "togunwa_full"), **m3})

    else:  # N2K
        # Existing: N2K (Kermany test)
        m, _ = run_inference_and_save(model, kermany_test_dataset, kermany_test_paths, ckpt.stem, "kermany_test")
        records.append({**info, "target": "kermany_test",
                         "condition": RES.derive_condition("N2K", "kermany_test"), **m})

        # N2N in-domain baseline (seed-matched Togunwa holdout, doubly
        # out-of-sample: excluded before kfold_splits, never used for training
        # or early-stopping/model-selection)
        holdout_ds, holdout_paths = build_togunwa_holdout_ds(seed)
        m2, _ = run_inference_and_save(model, holdout_ds, holdout_paths, ckpt.stem, "togunwa_holdout")
        records.append({**info, "target": "togunwa_holdout",
                         "condition": RES.derive_condition("N2K", "togunwa_holdout"), **m2})

    print(f"{ckpt.stem[:45]:45s} done")

summary = pd.DataFrame(records)
cols = ["arch", "direction", "condition", "size", "seed", "fold", "target",
        "n", "n_positive", "auroc", "specificity_at_95_sens", "ece", "ece_adaptive",
        "mce", "brier", "n_bins_used"]
cols = [c for c in cols if c in summary.columns]
summary = summary[cols]

summary_path = config.RESULTS_DIR / "04_inference_summary_extended.csv"
summary.to_csv(summary_path, index=False)

print(f"\nSaved extended summary ({len(summary)} rows) to {summary_path.name}")
print("\n--- Mean AUROC by condition ---")
print(summary.groupby("condition")["auroc"].agg(["mean", "std", "count"]).round(3).to_string())



Found 96 checkpoint file(s) for evaluation.
efficientnet_b0__K2N__size100__seed42         done
efficientnet_b0__K2N__size100__seed43         done
efficientnet_b0__K2N__size100__seed44         done
efficientnet_b0__K2N__size100__seed45         done
efficientnet_b0__K2N__size100__seed46         done
efficientnet_b0__K2N__size100__seed47         done
efficientnet_b0__K2N__size190__seed42         done
efficientnet_b0__K2N__size190__seed43         done
efficientnet_b0__K2N__size190__seed44         done
efficientnet_b0__K2N__size190__seed45         done
efficientnet_b0__K2N__size190__seed46         done
efficientnet_b0__K2N__size190__seed47         done
efficientnet_b0__K2N__size50__seed42          done
efficientnet_b0__K2N__size50__seed43          done
efficientnet_b0__K2N__size50__seed44          done
efficientnet_b0__K2N__size50__seed45          done
efficientnet_b0__K2N__size50__seed46          done
efficientnet_b0__K2N__size50__seed47          done
efficientnet_b0__N2K__size190__seed42

**Next:** notebook 05 (discrimination, SQ1).